# Fact credit_bureau_reports
1. read the silver credit_bureau_reports table
2. join with dim_customer, dim_date
3. select the required columns
- Measures: credit_score, external_active_loans, external_overdue_amount
- Foreign Keys: customer_sk, date_key
- Degenerate Dimensions: risk_grade
4. write the transformed data to gold fact_credit_bureau_reports table.

In [0]:
#Imports
from pyspark.sql.functions import col

In [0]:
credit_bureau_reports_df = spark.read.table("neo_bank.bronze.credit_bureau_reports")
customers_df = spark.read.table("neo_bank.gold.dim_customers")
date_df = spark.read.table("neo_bank.gold.dim_date")


In [0]:
credit_bureau_reports_df = credit_bureau_reports_df.select(
    col("customer_id").cast("int"),
    col("credit_score").cast("int"),
    col("risk_grade"),
    col("external_active_loans").cast("int"),
    col("external_overdue_amount").cast("int"),
    col("bureau_pull_date").cast("date")
)


In [0]:
fact_credit_bureau_reports_df = (
    credit_bureau_reports_df.alias("cr")
    .join(
        customers_df.alias("c"),
        col("cr.customer_id")==col("c.customer_id"),
        "left"
    )
    .join(
        date_df.alias("d"),
        col("cr.bureau_pull_date")==col("d.full_date"),
        "left"
    )
)

In [0]:
fact_credit_bureau_reports_df = fact_credit_bureau_reports_df.select(
    col("c.customer_sk"),
    col("cr.credit_score"),
    col("cr.risk_grade"),
    col("cr.external_active_loans"),
    col("cr.external_overdue_amount"),
    col("d.date_key").alias("bureau_pull_date_key")
)


In [0]:
(
    fact_credit_bureau_reports_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("neo_bank.gold.fact_credit_bureau_reports")
)

In [0]:
%sql
select * from neo_bank.gold.fact_credit_bureau_reports